## Required Imports

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
from torchvision.datasets import MNIST  # using MNIST for simple demonstration
from sklearn.decomposition import PCA

## Data Transforms

In [2]:
transform = transforms.Compose(
    [
        transforms.Resize((32, 32)), # for MNIST dataset, we resize the image to 32x32 for convenience purpose
        transforms.ToTensor(), # convert the image to a tensor  
    ]
)

# load the MNIST dataset
train_dataset = MNIST(root='../../data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='../../data', train=False, download=True, transform=transform)

## Set the Device

In [3]:
# define the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Exploring the Data

In [4]:
for image, label in train_dataset:
    print('-'*100)
    print('Basic Image Stats:')
    print(f'Image shape: {image.shape}.')
    print(f'Label: {label}.')
    print(f'Image min: {image.min()}.')
    print(f'Image max: {image.max()}.')
    print('-'*100)
    print(f'Image pixels:\n {image}.')
    print('-'*100)
    break

----------------------------------------------------------------------------------------------------
Basic Image Stats:
Image shape: torch.Size([1, 32, 32]).
Label: 5.
Image min: 0.0.
Image max: 0.9921568632125854.
----------------------------------------------------------------------------------------------------
Image pixels:
 tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]]).
----------------------------------------------------------------------------------------------------


## The Vanilla Autoencoder

In [5]:
class Autoencoder(nn.Module):
    """A simple autoencoder model. The encoder network is a stack of linear layers with ReLU activation functions
    as nonlinearities. The autoencoder network is trained to learn a compressed representation of the data.
    """
    def __init__(self, bottleneck_size: int=2):
        super(Autoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32 * 32),
            nn.Sigmoid() # ensure the output is in the range [0, 1] for visualization and reconstruction
                         # to ensure the output is in the range [-1, 1], we can use nn.Tanh() instead of nn.Sigmoid()
        )
    
    def encode(self, x):
        """Encoder function to encode the image to a latent space.
        """
        return self.encoder(x)
    
    def decode(self, x):
        """Decoder function to decode the latent space to an image.
        """
        x = self.decoder(x)
        x = x.reshape(-1, 1, 32, 32) # reshape the output to the original image shape
        return x
    
    def forward(self, x):
        b, c, h, w = x.shape

        x = x.flatten(start_dim=1) # flatten the image to a 1D vector starting from the second dimension

        encoded = self.encode(x) # encode the image to a latent space
        decoded = self.decode(encoded) # decode the latent space to an image

        return encoded, decoded

### A Simple Forward Pass

In [6]:
model = Autoencoder()
rand = torch.randn(2, 1, 32, 32)
model(rand)

(tensor([[ 0.1850, -0.0679],
         [ 0.1423, -0.0739]], grad_fn=<AddmmBackward0>),
 tensor([[[[0.5133, 0.5013, 0.4858,  ..., 0.5019, 0.5252, 0.4761],
           [0.4655, 0.4868, 0.5114,  ..., 0.4991, 0.4646, 0.4952],
           [0.5176, 0.5037, 0.4566,  ..., 0.5207, 0.4880, 0.5076],
           ...,
           [0.4939, 0.5135, 0.4884,  ..., 0.4742, 0.5212, 0.4658],
           [0.4581, 0.4736, 0.5332,  ..., 0.5191, 0.4889, 0.5193],
           [0.4947, 0.4974, 0.5099,  ..., 0.4930, 0.4723, 0.5096]]],
 
 
         [[[0.5133, 0.5015, 0.4863,  ..., 0.5018, 0.5249, 0.4761],
           [0.4650, 0.4866, 0.5117,  ..., 0.4992, 0.4645, 0.4955],
           [0.5178, 0.5038, 0.4560,  ..., 0.5205, 0.4877, 0.5076],
           ...,
           [0.4938, 0.5132, 0.4882,  ..., 0.4749, 0.5218, 0.4658],
           [0.4569, 0.4732, 0.5333,  ..., 0.5195, 0.4884, 0.5193],
           [0.4945, 0.4974, 0.5096,  ..., 0.4936, 0.4718, 0.5098]]]],
        grad_fn=<ViewBackward0>))

## Training

In [17]:
def train(
    model: nn.Module,
    train_dataset: Dataset,
    test_dataset: Dataset,
    batch_size: int,
    training_iterations: int,
    eval_iterations: int
    ):

        print("Training the model...")
        print(model)

        model  = model.to(device)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        optimizer = optim.Adam(model.parameters(), lr=0.0005)

        train_loss = []
        eval_loss = []
        train_losses = []
        eval_losses = []

        encoded_data_per_eval = []

        pbar = tqdm(range(training_iterations), desc="Training")
        last_eval_loss = None

        train = True

        global_step = 0

        while train:
            model.train()
            for images, labels in train_loader:
                images = images.to(device)

                optimizer.zero_grad()
                
                encoded, recon = model(images)

                loss = torch.mean((images - recon) ** 2) # compute the mean squared error
                train_loss.append(loss.item())

                loss.backward()
                optimizer.step()

                if global_step % eval_iterations == 0:
                    model.eval()
                    encoded_evals = []

                    for images, labels in test_loader:

                        with torch.no_grad():
                            images = images.to(device)
                        
                        encoded, recon = model(images)

                        loss = torch.mean((images - recon) ** 2)
                        eval_loss.append(loss.item())

                        # store the encoded data and the labels
                        encoded, labels = encoded.cpu().flatten(1), labels.reshape(-1, 1)
                        encoded_evals.append(torch.cat((encoded, labels), axis=-1))
                    
                    encoded_data_per_eval.append(torch.concatenate(encoded_evals).detach())
                    
                    train_loss = np.mean(train_loss)
                    eval_loss = np.mean(eval_loss)
                    last_eval_loss = eval_loss
                    
                    train_losses.append(train_loss)
                    eval_losses.append(eval_loss)

                    print(f'training loss: {train_loss:.4f}, eval loss: {eval_loss:.4f}')
                    
                    # reset the loss values for the next evaluation
                    train_loss = []
                    eval_loss = []
                
                global_step += 1
                pbar.update(1)

                if global_step >= training_iterations:
                    print("Training complete.")
                    train = False
                    break
        
        # store the encoded data as numpy arrays for each evaluation
        encoded_data_per_eval = [i.numpy() for i in encoded_data_per_eval]  # convert tensors to numpy (avoids np.array deprecation with PyTorch)

        return model, train_losses, eval_losses, encoded_data_per_eval

In [19]:
# start with a fresh model
model = Autoencoder(bottleneck_size=2)

In [20]:
model, train_losses, eval_losses, encoded_data_per_eval = train(
    model,
    train_dataset,
    test_dataset,
    batch_size=64,
    training_iterations=5000,
    eval_iterations=250)

Training the model...
Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=2, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=1024, bias=True)
    (7): Sigmoid()
  )
)


Training:   0%|          | 0/5000 [00:00<?, ?it/s]

training loss: 0.2206, eval loss: 0.2200
training loss: 0.0787, eval loss: 0.0524
training loss: 0.0491, eval loss: 0.0469
training loss: 0.0458, eval loss: 0.0454
training loss: 0.0447, eval loss: 0.0444
training loss: 0.0437, eval loss: 0.0436
training loss: 0.0430, eval loss: 0.0428
training loss: 0.0423, eval loss: 0.0422
training loss: 0.0418, eval loss: 0.0418
training loss: 0.0411, eval loss: 0.0413
training loss: 0.0408, eval loss: 0.0406
training loss: 0.0401, eval loss: 0.0400
training loss: 0.0391, eval loss: 0.0391
training loss: 0.0385, eval loss: 0.0382
training loss: 0.0377, eval loss: 0.0374
training loss: 0.0371, eval loss: 0.0370
training loss: 0.0366, eval loss: 0.0365
training loss: 0.0363, eval loss: 0.0361
training loss: 0.0361, eval loss: 0.0357
training loss: 0.0356, eval loss: 0.0357
Training complete.
